# 18 — Plan B: Reference Library + Decision Engine

Notebook 18 starts Plan B.

## Final objective

For a proposed/new bridge, the user provides a geographic location and available project parameters such as:

- latitude
- longitude
- length
- width
- DTV
- optionally material
- optionally Bauwerksart

The engine finds comparable existing bridges and separates:

1. **Similarity Score**
2. **Performance / Condition Evidence**
3. **Combined Decision Score**

The output is a transparent Bauwerksart recommendation.

## Boundary

The frozen 86-predictor condition model is not retrained, refitted, tuned, or modified.

Length and width are used in the Plan B similarity layer, not inserted into the frozen condition-model contract.

## 01 — Inputs and outputs

Only the current project roots are used.

```text
C:\Datenanalyse\final Project\Dataset_PlanA-B
C:\Datenanalyse\final Project\Output_PlanA-B
```

Notebook 17 already created the WGS84-ready map dataset. Therefore Notebook 18 uses that output as the geographic reference source instead of trying to reconstruct CRS information again.

Inputs:

```text
Output_PlanA-B/
├── 15_Plan_A_Current_Bridge_Condition/
│   └── plan_a_current_condition.parquet
├── 16_Plan_A_Future_Condition/
│   └── plan_a_future_condition.parquet
└── 17_Plan_A_Germany_Web_Map/
    └── plan_a_map_data.parquet
```

Outputs:

```text
Output_PlanA-B/
└── 18_Plan_B_Reference_Library/
    ├── plan_b_reference_library.parquet
    ├── plan_b_reference_library.csv
    ├── plan_b_reference_summary.csv
    └── 18_plan_b_reference_manifest.json
```

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

MAP17_INPUT = (
    OUTPUT_ROOT
    / "17_Plan_A_Germany_Web_Map"
    / "plan_a_map_data.parquet"
)
CURRENT_INPUT = (
    OUTPUT_ROOT
    / "15_Plan_A_Current_Bridge_Condition"
    / "plan_a_current_condition.parquet"
)
FUTURE_INPUT = (
    OUTPUT_ROOT
    / "16_Plan_A_Future_Condition"
    / "plan_a_future_condition.parquet"
)

OUTPUT_DIR = OUTPUT_ROOT / "18_Plan_B_Reference_Library"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LIBRARY_PARQUET = OUTPUT_DIR / "plan_b_reference_library.parquet"
LIBRARY_CSV = OUTPUT_DIR / "plan_b_reference_library.csv"
SUMMARY_CSV = OUTPUT_DIR / "plan_b_reference_summary.csv"
MANIFEST_JSON = OUTPUT_DIR / "18_plan_b_reference_manifest.json"

for p in [MAP17_INPUT, CURRENT_INPUT, FUTURE_INPUT]:
    if not p.exists():
        raise FileNotFoundError(f"Required current-project input not found:\n{p}")

print("[PASS] current Plan B inputs exist")
print("MAP17_INPUT:", MAP17_INPUT)

[PASS] current Plan B inputs exist
MAP17_INPUT: C:\Datenanalyse\final Project\Output_PlanA-B\17_Plan_A_Germany_Web_Map\plan_a_map_data.parquet


In [2]:
map17 = pd.read_parquet(MAP17_INPUT)
current = pd.read_parquet(CURRENT_INPUT)
future = pd.read_parquet(FUTURE_INPUT)

for name, df in [("map17", map17), ("current", current), ("future", future)]:
    if "bridge_id" not in df.columns:
        raise KeyError(f"{name} is missing bridge_id")
    df["bridge_id"] = df["bridge_id"].astype("string").str.strip()
    if not df["bridge_id"].is_unique:
        raise ValueError(f"{name} has duplicate bridge_id values")

for name, df in [("map17", map17), ("current", current), ("future", future)]:
    if len(df) != 52214:
        raise ValueError(f"{name}: expected 52,214 rows, found {len(df)}")

required_map = [
    "bridge_id","latitude","longitude","bauwerksart_text","baujahr",
    "age_years","laenge","breite","baustoffklasse",
    "traffic_dtv_latest","traffic_dtv_mean",
    "zustandsnote","gis_ort","gis_kreis","gis_bundesland"
]
required_current = [
    "bridge_id","zustandsnote_observed","zustandsnote_predicted"
]
required_future = [
    "bridge_id","zustandsnote_tplus_10y",
    "zustandsnote_tplus_25y","zustandsnote_tplus_50y"
]

for c in required_map:
    if c not in map17.columns:
        raise KeyError(f"Notebook 17 map output missing: {c}")
for c in required_current:
    if c not in current.columns:
        raise KeyError(f"Notebook 15 output missing: {c}")
for c in required_future:
    if c not in future.columns:
        raise KeyError(f"Notebook 16 output missing: {c}")

reference = (
    map17[required_map]
    .merge(current[required_current], on="bridge_id", how="left", validate="one_to_one")
    .merge(future[required_future], on="bridge_id", how="left", validate="one_to_one")
)

for c in [
    "latitude","longitude","laenge","breite",
    "traffic_dtv_latest","traffic_dtv_mean",
    "zustandsnote_observed","zustandsnote_predicted",
    "age_years","zustandsnote_tplus_10y",
    "zustandsnote_tplus_25y","zustandsnote_tplus_50y"
]:
    reference[c] = pd.to_numeric(reference[c], errors="coerce")

reference["dtv_reference"] = (
    reference["traffic_dtv_latest"]
    .combine_first(reference["traffic_dtv_mean"])
)

if len(reference) != 52214 or reference["bridge_id"].nunique() != 52214:
    raise ValueError("Plan B reference population/uniqueness gate failed.")

print("[PASS] reference population: 52,214")
print("[PASS] unique bridge_id: 52,214")

[PASS] reference population: 52,214
[PASS] unique bridge_id: 52,214


## 02 — Transparent similarity configuration

Initial agreed weights:

```text
Bauwerksart   30%
Baustoff      20%
Länge         15%
Breite        10%
DTV           15%
Distanz       10%
```

These are transparent initial weights, not statistically optimized weights.

If a user does not provide an optional field, that component is excluded and the available weights are renormalized.

Missing DTV is never interpreted as zero.

In [3]:
WEIGHTS = {
    "bauwerksart": 0.30,
    "baustoff": 0.20,
    "length": 0.15,
    "width": 0.10,
    "dtv": 0.15,
    "distance": 0.10,
}

DISTANCE_SCALE_KM = 50.0

length_scale = max(float(reference["laenge"].dropna().quantile(0.75)), 1.0)
width_scale = max(float(reference["breite"].dropna().quantile(0.75)), 1.0)

print(WEIGHTS)
print("length_scale:", length_scale)
print("width_scale:", width_scale)
print("distance_scale_km:", DISTANCE_SCALE_KM)
print("[PASS] similarity configuration")

{'bauwerksart': 0.3, 'baustoff': 0.2, 'length': 0.15, 'width': 0.1, 'dtv': 0.15, 'distance': 0.1}
length_scale: 45.0
width_scale: 18.0
distance_scale_km: 50.0
[PASS] similarity configuration


In [4]:
def haversine_km(lat, lon, ref_lat, ref_lon):
    lat1 = np.radians(float(lat))
    lon1 = np.radians(float(lon))
    lat2 = np.radians(np.asarray(ref_lat, dtype=float))
    lon2 = np.radians(np.asarray(ref_lon, dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat/2.0)**2
        + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    )
    return 6371.0088 * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def cat_sim(series, value):
    if value is None or pd.isna(value):
        return pd.Series(np.nan, index=series.index)
    return series.astype("string").eq(str(value)).astype(float)

def num_sim(series, value, scale):
    if value is None or pd.isna(value):
        return pd.Series(np.nan, index=series.index)
    x = pd.to_numeric(series, errors="coerce")
    return pd.Series(
        np.where(x.notna(), np.exp(-np.abs(x-float(value))/scale), np.nan),
        index=series.index
    )

def dtv_sim(series, value):
    if value is None or pd.isna(value):
        return pd.Series(np.nan, index=series.index)
    x = pd.to_numeric(series, errors="coerce")
    v = float(value)
    denom = np.maximum(np.maximum(np.abs(x), abs(v)), 1.0)
    s = np.clip(1.0 - np.abs(x-v)/denom, 0.0, 1.0)
    return pd.Series(np.where(x.notna(), s, np.nan), index=series.index)

print("[PASS] similarity functions")

[PASS] similarity functions


## 03 — Performance / condition evidence

Condition is evidence, not a replacement for similarity.

For each existing bridge:

- observed `zustandsnote` is used when available;
- otherwise the current frozen-model prediction is used.

The project condition scale is 1–4, so the evidence score is normalized to `[0,1]`:

```text
1.0 condition → 1.0 evidence
4.0 condition → 0.0 evidence
```

Future condition values remain available in the library for later Plan B scenario analysis.

In [5]:
condition = reference["zustandsnote_observed"].combine_first(
    reference["zustandsnote_predicted"]
)

reference["performance_score"] = np.clip(
    1.0 - (condition - 1.0) / 3.0,
    0.0,
    1.0
)

print(
    "Performance evidence coverage:",
    int(reference["performance_score"].notna().sum()),
    "/ 52214"
)
print("[PASS] performance evidence")

Performance evidence coverage: 52214 / 52214
[PASS] performance evidence


## 04 — Operational decision function

The engine evaluates the existing reference population.

For a proposed bridge:

```python
result, type_summary = score_proposed_bridge(
    latitude=...,
    longitude=...,
    length=...,
    width=...,
    dtv=...,
    baustoff=...,
)
```

If `bauwerksart` is omitted, the engine does not assume one. It evaluates existing Bauwerksarten through the reference cohort and returns a ranked type-level evidence table.

The combined score is:

```text
0.80 × Similarity Score
+
0.20 × Performance Score
```

This is a transparent decision-support score, not a probability and not an FEM design result.

In [6]:
def score_proposed_bridge(
    latitude,
    longitude,
    length,
    width,
    dtv=None,
    baustoff=None,
    bauwerksart=None,
    top_k_references=25,
    min_references_per_type=5,
):
    work = reference.copy()

    ref_lat = work["latitude"]
    ref_lon = work["longitude"]

    valid_geo = (
        pd.notna(latitude) and pd.notna(longitude)
        and ref_lat.notna() & ref_lon.notna()
    )

    work["distance_km"] = np.nan
    if pd.notna(latitude) and pd.notna(longitude):
        mask = ref_lat.notna() & ref_lon.notna()
        work.loc[mask, "distance_km"] = haversine_km(
            latitude,
            longitude,
            ref_lat.loc[mask].to_numpy(),
            ref_lon.loc[mask].to_numpy(),
        )

    work["sim_bauwerksart"] = cat_sim(work["bauwerksart_text"], bauwerksart)
    work["sim_baustoff"] = cat_sim(work["baustoffklasse"], baustoff)
    work["sim_length"] = num_sim(work["laenge"], length, length_scale)
    work["sim_width"] = num_sim(work["breite"], width, width_scale)
    work["sim_dtv"] = dtv_sim(work["dtv_reference"], dtv)
    work["sim_distance"] = (
        np.exp(-work["distance_km"] / DISTANCE_SCALE_KM)
        .where(work["distance_km"].notna())
    )

    # If Bauwerksart is not supplied, it is intentionally not a similarity
    # input. Type recommendation then comes from cohort aggregation.
    components = {
        "bauwerksart": "sim_bauwerksart",
        "baustoff": "sim_baustoff",
        "length": "sim_length",
        "width": "sim_width",
        "dtv": "sim_dtv",
        "distance": "sim_distance",
    }

    numerator = np.zeros(len(work))
    denominator = np.zeros(len(work))

    for name, col in components.items():
        available = work[col].notna().to_numpy()
        w = WEIGHTS[name]
        numerator += np.where(
            available,
            work[col].fillna(0).to_numpy() * w,
            0.0
        )
        denominator += np.where(available, w, 0.0)

    work["similarity_score"] = np.where(
        denominator > 0,
        numerator / denominator,
        np.nan
    )

    work = work[work["similarity_score"].notna()].copy()

    if work.empty:
        raise ValueError("No reference bridge can be scored.")

    work["decision_score"] = (
        0.80 * work["similarity_score"]
        + 0.20 * work["performance_score"].fillna(0.0)
    )

    work = work.sort_values(
        ["decision_score", "similarity_score"],
        ascending=[False, False]
    )

    top_refs = work.head(int(top_k_references)).copy()

    type_summary = (
        work.groupby("bauwerksart_text", dropna=False)
        .agg(
            reference_count=("bridge_id", "count"),
            mean_similarity=("similarity_score", "mean"),
            mean_performance=("performance_score", "mean"),
            mean_decision_score=("decision_score", "mean"),
            max_decision_score=("decision_score", "max"),
        )
        .reset_index()
    )

    type_summary["eligible_reference_count"] = (
        type_summary["reference_count"] >= int(min_references_per_type)
    )

    type_summary = type_summary.sort_values(
        ["eligible_reference_count", "mean_decision_score", "reference_count"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    if not type_summary["eligible_reference_count"].any():
        raise ValueError("No Bauwerksart has the required reference cohort size.")

    return top_refs, type_summary

print("[PASS] Plan B decision engine defined")

[PASS] Plan B decision engine defined


## 05 — Export the reusable reference library

The library is the input to the later Plan B user interface.

It contains one row per existing bridge and enough information to reproduce the similarity and performance layers.

In [7]:
library_columns = [
    "bridge_id","latitude","longitude",
    "bauwerksart_text","baustoffklasse","baujahr","age_years",
    "laenge","breite","dtv_reference",
    "traffic_dtv_latest","traffic_dtv_mean",
    "zustandsnote_observed","zustandsnote_predicted",
    "zustandsnote_tplus_10y","zustandsnote_tplus_25y",
    "zustandsnote_tplus_50y","performance_score",
    "gis_ort","gis_kreis","gis_bundesland"
]

library = reference[library_columns].copy()

library.to_parquet(LIBRARY_PARQUET, index=False)
library.to_csv(LIBRARY_CSV, index=False)

summary = pd.DataFrame([
    {"metric":"reference_bridge_count","value":len(library)},
    {"metric":"unique_bridge_id","value":library["bridge_id"].nunique()},
    {"metric":"wgs84_coverage","value":int((library["latitude"].notna() & library["longitude"].notna()).sum())},
    {"metric":"dtv_coverage","value":int(library["dtv_reference"].notna().sum())},
    {"metric":"condition_evidence_coverage","value":int(library["performance_score"].notna().sum())},
    {"metric":"bauwerksart_count","value":library["bauwerksart_text"].nunique()},
])

summary.to_csv(SUMMARY_CSV, index=False)

print("[PASS] library parquet:", LIBRARY_PARQUET)
print("[PASS] library csv:", LIBRARY_CSV)

[PASS] library parquet: C:\Datenanalyse\final Project\Output_PlanA-B\18_Plan_B_Reference_Library\plan_b_reference_library.parquet
[PASS] library csv: C:\Datenanalyse\final Project\Output_PlanA-B\18_Plan_B_Reference_Library\plan_b_reference_library.csv


In [8]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "notebook": "18_Plan_B_Reference_Library_and_Decision_Engine",
    "reference_population": 52214,
    "inputs": {
        "notebook_17_map_data": str(MAP17_INPUT),
        "notebook_15_current_condition": str(CURRENT_INPUT),
        "notebook_16_future_condition": str(FUTURE_INPUT),
    },
    "similarity_weights": WEIGHTS,
    "weight_status": "transparent_initial_weights_not_statistically_optimized",
    "distance_scale_km": DISTANCE_SCALE_KM,
    "length_scale": length_scale,
    "width_scale": width_scale,
    "combined_decision_formula": "0.80 * similarity_score + 0.20 * performance_score",
    "frozen_condition_model_changed": False,
    "frozen_condition_model_retrained": False,
    "fem_performed": False,
    "outputs": {
        "library_parquet": str(LIBRARY_PARQUET),
        "library_csv": str(LIBRARY_CSV),
        "summary": str(SUMMARY_CSV),
    }
}

MANIFEST_JSON.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("[PASS] manifest:", MANIFEST_JSON)

[PASS] manifest: C:\Datenanalyse\final Project\Output_PlanA-B\18_Plan_B_Reference_Library\18_plan_b_reference_manifest.json


## 06 — Final Notebook 18 gate

Notebook 18 is complete when the full existing-bridge population is available as a Plan B reference library.

No recommendation is hard-coded into the library export. Recommendation is generated by the decision function from the user-provided scenario.

In [9]:
final_checks = {
    "reference_count_52214": len(library) == 52214,
    "unique_bridge_id": library["bridge_id"].nunique() == 52214,
    "latitude_present": "latitude" in library.columns,
    "longitude_present": "longitude" in library.columns,
    "bauwerksart_present": library["bauwerksart_text"].notna().all(),
    "baustoff_present": library["baustoffklasse"].notna().all(),
    "library_parquet_exists": LIBRARY_PARQUET.exists(),
    "library_csv_exists": LIBRARY_CSV.exists(),
    "summary_exists": SUMMARY_CSV.exists(),
    "manifest_exists": MANIFEST_JSON.exists(),
}

print("FINAL NOTEBOOK 18 GATE")
for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("Notebook 18 final gate failed.")

print()
print("18 STATUS: COMPLETE")
print("Reference library:", LIBRARY_PARQUET)

FINAL NOTEBOOK 18 GATE
[PASS] reference_count_52214
[PASS] unique_bridge_id
[PASS] latitude_present
[PASS] longitude_present
[PASS] bauwerksart_present
[PASS] baustoff_present
[PASS] library_parquet_exists
[PASS] library_csv_exists
[PASS] summary_exists
[PASS] manifest_exists

18 STATUS: COMPLETE
Reference library: C:\Datenanalyse\final Project\Output_PlanA-B\18_Plan_B_Reference_Library\plan_b_reference_library.parquet
